In [1]:
import os, json, random, re
from datasets import load_dataset

/workspace/venvs/qwen/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
OUT_DIR = "/root/data/eval_data"
SEED = 0
N_PER_DATASET = 5000   
random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)

def write_jsonl(path, rows):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

def mc_prompt(question, choices):
    """
    统一的多选题 prompt：要求输出单个选项字母。
    """
    letters = [chr(ord('A') + i) for i in range(len(choices))]
    options = "\n".join([f"{letters[i]}. {choices[i]}" for i in range(len(choices))])
    prompt = (
        "You are given a multiple-choice question. "
        "Choose the best answer. Reply with only the letter (A, B, C, ...).\n\n"
        f"Question:\n{question}\n\nChoices:\n{options}\n\nAnswer:"
    )
    return prompt

In [3]:
def build_hellaswag(n=N_PER_DATASET):
    ds = load_dataset("hellaswag", split="validation")
    idxs = list(range(len(ds)))
    random.shuffle(idxs)

    rows = []
    for i in idxs:
        ex = ds[i]
        # HellaSwag: ctx + endings[4], label is correct ending index
        context = (ex.get("ctx") or "").strip()
        endings = ex.get("endings") or []
        label = ex.get("label")
        if not context or not endings or label is None:
            continue
        if len(endings) != 4:
            continue

        question = context
        choices = [e.strip() for e in endings]
        prompt = mc_prompt(question, choices)

        rows.append({
            "id": f"hellaswag_val_{ex.get('ind', i)}",
            "dataset": "hellaswag",
            "messages": [{"role": "user", "content": prompt}],
            "choices": choices,
            "answer_index": int(label),
        })
        if len(rows) >= n:
            break

    out_path = os.path.join(OUT_DIR, f"hellaswag_val_{len(rows)}.jsonl")
    write_jsonl(out_path, rows)
    return out_path, len(rows)

path_hs, n_hs = build_hellaswag()
path_hs, n_hs

('/root/data/eval_data/hellaswag_val_5000.jsonl', 5000)

In [4]:
def build_arc_challenge(n=N_PER_DATASET):
    ds = load_dataset("ai2_arc", "ARC-Challenge", split="test")
    idxs = list(range(len(ds)))
    random.shuffle(idxs)

    rows = []
    for i in idxs:
        ex = ds[i]
        q = ex.get("question", "")
        choices_obj = ex.get("choices", {})
        labels = choices_obj.get("label", [])
        texts  = choices_obj.get("text", [])
        ans_key = ex.get("answerKey", None)

        if not q or not labels or not texts or ans_key is None:
            continue

        # ARC labels are often 'A','B','C','D' but sometimes more.
        # We'll sort by label to make stable order.
        pairs = [(lab.strip(), txt.strip()) for lab, txt in zip(labels, texts) if lab and txt]
        pairs = sorted(pairs, key=lambda x: x[0])

        # Map answerKey to index
        label_to_idx = {lab: idx for idx, (lab, _) in enumerate(pairs)}
        if ans_key not in label_to_idx:
            continue

        choices = [t for _, t in pairs]
        prompt = mc_prompt(q.strip(), choices)

        rows.append({
            "id": f"arc_challenge_test_{i}",
            "dataset": "arc_challenge",
            "messages": [{"role": "user", "content": prompt}],
            "choices": choices,
            "answer_index": int(label_to_idx[ans_key]),
        })
        if len(rows) >= n:
            break

    out_path = os.path.join(OUT_DIR, f"arc_challenge_test_{len(rows)}.jsonl")
    write_jsonl(out_path, rows)
    return out_path, len(rows)

path_arc, n_arc = build_arc_challenge()
path_arc, n_arc

('/root/data/eval_data/arc_challenge_test_1172.jsonl', 1172)

In [5]:
def build_winogrande(n=N_PER_DATASET):
    ds = load_dataset("winogrande", "winogrande_xl", split="validation")
    idxs = list(range(len(ds)))
    random.shuffle(idxs)

    rows = []
    for i in idxs:
        ex = ds[i]
        sent = (ex.get("sentence") or "").strip()
        opt1 = (ex.get("option1") or "").strip()
        opt2 = (ex.get("option2") or "").strip()
        ans = ex.get("answer", None)  # "1" or "2"

        if not sent or not opt1 or not opt2 or ans is None:
            continue
        if "_" not in sent:
            continue

        question = sent.replace("_", "____")
        choices = [opt1, opt2]
        prompt = mc_prompt(question, choices)

        rows.append({
            "id": f"winogrande_val_{ex.get('qID', i)}",
            "dataset": "winogrande",
            "messages": [{"role": "user", "content": prompt}],
            "choices": choices,
            "answer_index": int(ans) - 1,
        })
        if len(rows) >= n:
            break

    out_path = os.path.join(OUT_DIR, f"winogrande_val_{len(rows)}.jsonl")
    write_jsonl(out_path, rows)
    return out_path, len(rows)

path_wg, n_wg = build_winogrande()
path_wg, n_wg

('/root/data/eval_data/winogrande_val_1267.jsonl', 1267)

In [6]:
def build_truthfulqa_mc(n=N_PER_DATASET):
    ds = load_dataset("truthful_qa", "multiple_choice", split="validation")
    idxs = list(range(len(ds)))
    random.shuffle(idxs)

    rows = []
    for i in idxs:
        ex = ds[i]
        q = (ex.get("question") or "").strip()
        choices = ex.get("mc1_targets", {}).get("choices", None)
        labels  = ex.get("mc1_targets", {}).get("labels", None)

        if not q or choices is None or labels is None:
            continue
        # In truthfulqa multiple_choice, mc1_targets.labels is usually 0/1 (one correct)
        if len(choices) < 2 or len(choices) != len(labels):
            continue

        # pick the (first) correct option
        try:
            ans_idx = labels.index(1)
        except ValueError:
            continue

        choices = [c.strip() for c in choices]
        prompt = mc_prompt(q, choices)

        rows.append({
            "id": f"truthfulqa_mc_val_{i}",
            "dataset": "truthfulqa_mc",
            "messages": [{"role": "user", "content": prompt}],
            "choices": choices,
            "answer_index": int(ans_idx),
        })
        if len(rows) >= n:
            break

    out_path = os.path.join(OUT_DIR, f"truthfulqa_mc_val_{len(rows)}.jsonl")
    write_jsonl(out_path, rows)
    return out_path, len(rows)

path_tqa, n_tqa = build_truthfulqa_mc()
path_tqa, n_tqa

('/root/data/eval_data/truthfulqa_mc_val_817.jsonl', 817)

In [7]:
manifest = [
    {"name": "hellaswag",      "path": path_hs,  "n": n_hs},
    {"name": "arc_challenge",  "path": path_arc, "n": n_arc},
    {"name": "winogrande",     "path": path_wg,  "n": n_wg},
    {"name": "truthfulqa_mc",  "path": path_tqa, "n": n_tqa},
]
manifest_path = os.path.join(OUT_DIR, "manifest.json")
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

manifest_path, manifest

('/root/data/eval_data/manifest.json',
 [{'name': 'hellaswag',
   'path': '/root/data/eval_data/hellaswag_val_5000.jsonl',
   'n': 5000},
  {'name': 'arc_challenge',
   'path': '/root/data/eval_data/arc_challenge_test_1172.jsonl',
   'n': 1172},
  {'name': 'winogrande',
   'path': '/root/data/eval_data/winogrande_val_1267.jsonl',
   'n': 1267},
  {'name': 'truthfulqa_mc',
   'path': '/root/data/eval_data/truthfulqa_mc_val_817.jsonl',
   'n': 817}])

In [8]:
import json

sample_path = manifest[0]["path"]
with open(sample_path, "r", encoding="utf-8") as f:
    line = f.readline()
ex = json.loads(line)
ex.keys(), ex["dataset"], ex["answer_index"], ex["choices"][:2], ex["messages"][0]["content"][:400]

(dict_keys(['id', 'dataset', 'messages', 'choices', 'answer_index']),
 'hellaswag',
 0,
 ['You will be splattering the paint onto your jeans and it is likely to splatter all around your jeans as well. Lay out your tarp or newspapers to cover the area you will be working on.',
  'You might want to put out a tarp or blanket for added protection. Do not make the area wet; any paint sitting around your work area could stain your floor.'],
 'You are given a multiple-choice question. Choose the best answer. Reply with only the letter (A, B, C, ...).\n\nQuestion:\n[header] How to paint jeans [title] Cover your work area. [step] Protect your surfaces from the paint or it may stain them. Make sure to cover a large area.\n\nChoices:\nA. You will be splattering the paint onto your jeans and it is likely to splatter all around your jeans as well. ')